# 3.15 Random Values — Homework (SFI Backend QA Test Garage)

Lesson: https://pages.opencodingsociety.com/python/random

Theme: QA engineer stress-testing an SFI automotive safety-spec backend using `random.randint(a, b)` (inclusive) and `random.choice(sequence)`.

College Board connection: `RANDOM(a, b)` → `random.randint(a, b)`, both inclusive. `RANDOM(1, 4)` possible outputs: `1, 2, 3, 4`.


## Popcorn Hack 1: Check One SFI Part Record

**Task (from https://pages.opencodingsociety.com/python/boolean/2027):** An SFI automotive record is complete when it has a product name, a racing category, a spec number, and an effective date. For this challenge, **Auto Racing** and **Drag Racing** are valid racing categories. Report the Boolean state of each requirement for the record below.

**Starter:**
```python
part = {
    "product_name": "Replacement Flywheels and Clutch Assemblies",
    "category": "Auto Racing",
    "spec_number": "1.1",
    "effective_date": "Nov. 9, 2001"
}
valid_categories = ["Auto Racing", "Drag Racing"]
```

**Prediction:** First run prints all `True` — name is non-empty, `Auto Racing` is in the valid list, spec and date are non-empty. Changing `category` to `Street Car` flips only `valid_category` to `False`.

**Reasoning rule:** Comparisons create Booleans (`!=`, `in`, `==`). `in valid_categories` is the Python form of `(category = "Auto Racing") OR (category = "Drag Racing")`.


In [ ]:
# Popcorn Hack 1 - Check SFI Part Fields
# Reference: https://pages.opencodingsociety.com/python/boolean/2027

part = {
    "product_name": "Replacement Flywheels and Clutch Assemblies",
    "category": "Auto Racing",
    "spec_number": "1.1",
    "effective_date": "Nov. 9, 2001"
}

valid_categories = ["Auto Racing", "Drag Racing"]

# Build and test your solution below.
# Each comparison evaluates to True / False.
has_product_name = part["product_name"] != ""
valid_category = part["category"] in valid_categories
has_spec_number = part["spec_number"] != ""
has_effective_date = part["effective_date"] != ""

print("Has product name:", has_product_name)
print("Valid category (Auto Racing OR Drag Racing):", valid_category)
print("Has spec number:", has_spec_number)
print("Has effective date:", has_effective_date)

# Changed-input check (in-class second run): Street Car is not accepted
part_changed = dict(part)
part_changed["category"] = "Street Car"
print("Changed category to Street Car -> valid:", part_changed["category"] in valid_categories)


**Output recorded:** `Has product name: True`, `Valid category: True`, `Has spec number: True`, `Has effective date: True`; changed-input run `Street Car -> valid: False`.

**Explanation:** All four checks are `True` for the starter record, so it is complete. `valid_category` uses `in ["Auto Racing", "Drag Racing"]`, matching the lesson's `OR` rule — either accepted value passes, anything else (`Street Car`) is `False`. This is the field-level Boolean check the Homework validator reuses with `and` / `not`.


## Popcorn Hack 2: Backend Create Rule

**Task (from https://pages.opencodingsociety.com/python/boolean/2027):** The backend receives the same style of SFI part record. A new database record should only be created when all required fields are valid **and** its spec number is not already stored. Build the final create decision and test it with at least one valid case and one rejected case.

**Prediction:** Starter record (`spec 1.1`, already in `["1.1", "2.1"]`) prints REJECT even though its fields are valid, because `NOT duplicate` fails. A second record with new spec `1.2` prints CREATE.

**Reasoning rule:** All requirements must pass -> `and`. Reject a duplicate -> `not duplicate_spec`. `Tech Talk 2` demo: `has_product_name and has_spec_number`, `or` for either-option, `not` to reverse.


In [ ]:
# Popcorn Hack 2 - Backend Create Rule
# Reference: https://pages.opencodingsociety.com/python/boolean/2027

part = {
    "product_name": "Replacement Flywheels and Clutch Assemblies",
    "category": "Auto Racing",
    "spec_number": "1.1",
    "effective_date": "Nov. 9, 2001"
}

valid_categories = ["Auto Racing", "Drag Racing"]
existing_spec_numbers = ["1.1", "2.1"]

# Build and test your solution below.
def can_create_record(p):
    has_product_name = p.get("product_name", "") != ""
    valid_category = p.get("category", "") in valid_categories
    has_spec_number = p.get("spec_number", "") != ""
    has_effective_date = p.get("effective_date", "") != ""
    duplicate_spec = p.get("spec_number", "") in existing_spec_numbers
    can_create = (
        has_product_name
        and valid_category
        and has_spec_number
        and has_effective_date
        and (not duplicate_spec)
    )
    return can_create

# Case 1: starter record -> duplicate spec 1.1 -> REJECT
can_create = can_create_record(part)
print("Starter (spec 1.1) can_create:", can_create)
if can_create:
    print("CREATE record in backend")
else:
    print("REJECT record: duplicate spec or invalid fields")

# Case 2: valid new record -> CREATE (tests the opposite path)
new_part = {
    "product_name": "Multiple Disc Clutch Assemblies",
    "category": "Drag Racing",
    "spec_number": "1.2",
    "effective_date": "Feb. 9, 2006"
}
can_create_new = can_create_record(new_part)
print("New part (spec 1.2) can_create:", can_create_new)
if can_create_new:
    print("CREATE record in backend")
else:
    print("REJECT record: duplicate spec or invalid fields")


**Output recorded:** `Starter (spec 1.1) can_create: False -> REJECT`; `New part (spec 1.2) can_create: True -> CREATE`.

**Explanation:** Starter fails only on `duplicate_spec = True`, so `... and (not True)` = `False`. The `1.2` record passes all five checks, so `and`-chain is `True`. Both branches of the `if/else` were exercised, as the instructions require (one valid, one rejected).


## Popcorn Hack 3: SFI Search Filter

**Task (from https://pages.opencodingsociety.com/python/boolean/2027):** An SFI backend search can match a record through either its product name or its exact spec number. Search results should only include records from accepted racing categories. Use the provided records and query to return only valid matches. Logic must still work when the query or record data changes.

**Predictions:** Query `"1.2"` matches exactly one record (`Multiple Disc Clutch Assemblies`, Drag Racing). Query `"Flywheel"` matches by name but only in accepted categories. Changing the query changes the result without changing the code.

**Reasoning rule:** Either match passes -> `or` for `(query in name) OR (query == spec)`. Category gate -> `and valid_category`. Combines to `(name_match or spec_match) and valid_category`.


In [ ]:
# Popcorn Hack 3 - SFI Search Filter
# Reference: https://pages.opencodingsociety.com/python/boolean/2027

query = "1.2"

records = [
    {
        "product_name": "Replacement Flywheels and Clutch Assemblies",
        "category": "Auto Racing",
        "spec_number": "1.1"
    },
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Drag Racing",
        "spec_number": "1.2"
    },
    {
        "product_name": "Racing Flywheel Record",
        "category": "Auto Racing",
        "spec_number": "2.1"
    }
]

valid_categories = ["Auto Racing", "Drag Racing"]

# Build and test your solution below.
def search_records(q):
    results = []
    for r in records:
        name_match = q in r["product_name"]
        spec_match = q == r["spec_number"]
        valid_category = r["category"] in valid_categories
        if (name_match or spec_match) and valid_category:
            results.append(r)
    return results

# Case 1: starter query from the lesson
matches = search_records(query)
print(f'Query "{query}" -> {len(matches)} match(es):')
for m in matches:
    print(" ", m["product_name"], "|", m["category"], "| spec", m["spec_number"])

# Case 2: changed query proves logic still works when data changes
query2 = "Flywheel"
matches2 = search_records(query2)
print(f'Query "{query2}" -> {len(matches2)} match(es):')
for m in matches2:
    print(" ", m["product_name"], "|", m["category"], "| spec", m["spec_number"])


**Output recorded:** Query `"1.2" -> 1 match (Multiple Disc Clutch Assemblies | Drag Racing | 1.2)`; Query `"Flywheel" -> 2 matches` (both in accepted categories).

**Explanation:** `(name_match or spec_match) and valid_category` is the whole filter. `"1.2"` hits only via `spec_match`; `"Flywheel"` hits via `name_match`. A `Street Car` record with the same name would still be excluded by the `and valid_category` gate, which is why the category check must stay inside the condition.


## In-class MCQ Check

I've added the text asked for me to add in the description. Here it is just incase:

```
MCQ 3.05: 4/4 | answers: A,C,A,B
```

## Homework Hack (independent): SFI Backend Validator

**Starter code (use as-is, build below it):** `existing_spec_numbers`, `valid_categories`, `test_records` with 3 candidate records.

**Reference:** https://pages.opencodingsociety.com/python/boolean/2027 — Boolean expressions (`==`, `!=`, `in`, `and`, `or`, `not`).

**Valid record rule (all must pass):**
- non-empty `product_name` **AND**
- `category` in `valid_categories` (`Auto Racing` **OR** `Drag Racing`) **AND**
- non-empty `spec_number` **AND**
- non-empty `effective_date` **AND**
- **NOT** duplicate (`spec_number` not in `existing_spec_numbers`)

**Random Values connection (3.15):** after the deterministic check of all 3 starter records, use `random.choice(test_records)` for 5 randomized QA draws and `random.randint(1, 4)` to show inclusive `RANDOM(a,b)` range. Same validator, random test driver.


In [ ]:
# Homework Hack - SFI Backend Validator

import random

existing_spec_numbers = ["1.1", "2.1"]
valid_categories = ["Auto Racing", "Drag Racing"]

test_records = [
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Auto Racing",
        "spec_number": "1.2",
        "effective_date": "Feb. 9, 2006"
    },
    {
        "product_name": "Replacement Flywheels",
        "category": "Street Car",
        "spec_number": "3.1",
        "effective_date": "Jan. 1, 2026"
    },
    {
        "product_name": "Existing Flywheel Record",
        "category": "Drag Racing",
        "spec_number": "1.1",
        "effective_date": "Jan. 1, 2026"
    }
]

# Build and test your validator below.
# Boolean reference: https://pages.opencodingsociety.com/python/boolean/2027


def is_valid_record(record):
    """Validate one SFI record. Returns (is_valid, checks_dict)."""
    has_product_name = record.get("product_name", "") != ""
    # OR logic: either accepted category passes
    valid_category = record.get("category", "") in valid_categories
    has_spec_number = record.get("spec_number", "") != ""
    has_effective_date = record.get("effective_date", "") != ""
    # duplicate if spec already in backend list
    duplicate_spec = record.get("spec_number", "") in existing_spec_numbers

    is_valid = (
        has_product_name
        and valid_category
        and has_spec_number
        and has_effective_date
        and (not duplicate_spec)
    )
    checks = {
        "has_product_name": has_product_name,
        "valid_category": valid_category,
        "has_spec_number": has_spec_number,
        "has_effective_date": has_effective_date,
        "duplicate_spec": duplicate_spec,
        "is_valid": is_valid,
    }
    return is_valid, checks


def verdict(record):
    is_valid, checks = is_valid_record(record)
    status = "ACCEPT RECORD" if is_valid else "REJECT RECORD"
    return status, checks


# 1) Deterministic check: every starter record, in order
print("== Deterministic validation (all 3 starter records) ==")
for r in test_records:
    status, checks = verdict(r)
    print(f"{r['product_name']} (spec {r['spec_number']}, {r['category']}) -> {status}")
    print(f"   checks: {checks}")

# 2) Random-values driver: same validator, random test order
# RANDOM(1,4) -> random.randint(1, 4), both endpoints inclusive
print("\n== Randomized QA (5 random draws via random.choice) ==")
for i in range(1, 6):
    pick = random.choice(test_records)  # one intact dict, fields stay together
    draw = random.randint(1, 4)  # inclusive demo: possible outputs {1,2,3,4}
    status, _ = verdict(pick)
    print(f"[Random test {i}] draw={draw} | {pick['product_name']} (spec {pick['spec_number']}) -> {status}")


### Homework validation tests

Seeded repeatability check + Boolean-logic assertions + `randint` range check. Expected: record `1.2` ACCEPTs, `3.1` REJECTs (bad category), `1.1` REJECTs (duplicate).


In [ ]:
# Validation: Boolean logic + random-values checks
import random

# 1) Boolean expectations on the exact starter data
status_12, _ = verdict(test_records[0])  # 1.2 / Auto Racing / new spec
status_31, _ = verdict(test_records[1])  # 3.1 / Street Car (invalid category)
status_11, _ = verdict(test_records[2])  # 1.1 / duplicate spec
assert status_12 == "ACCEPT RECORD", status_12
assert status_31 == "REJECT RECORD", status_31
assert status_11 == "REJECT RECORD", status_11

# 2) Compound-expression unit checks (and / or / not match the written rule)
assert is_valid_record(
    {"product_name": "X", "category": "Auto Racing",
     "spec_number": "9.9", "effective_date": "Jan. 1, 2026"}
)[0] is True
# empty name -> reject (has_product_name is False)
assert is_valid_record(
    {"product_name": "", "category": "Auto Racing",
     "spec_number": "9.9", "effective_date": "Jan. 1, 2026"}
)[0] is False
# Street Car -> reject (valid_category uses OR over accepted list)
assert is_valid_record(
    {"product_name": "X", "category": "Street Car",
     "spec_number": "9.9", "effective_date": "Jan. 1, 2026"}
)[0] is False
# duplicate spec -> reject (NOT duplicate_spec fails)
assert is_valid_record(
    {"product_name": "X", "category": "Drag Racing",
     "spec_number": "1.1", "effective_date": "Jan. 1, 2026"}
)[0] is False

# 3) Random-values checks: inclusive randint + intact choice
random.seed(0)
draws = [random.randint(1, 4) for _ in range(200)]
assert min(draws) == 1 and max(draws) == 4, draws
assert set(draws) == {1, 2, 3, 4}, set(draws)

random.seed(15)
picks = [random.choice(test_records) for _ in range(5)]
assert all(p in test_records for p in picks)
print("Random draws sample:", [p["spec_number"] for p in picks])

print("Validation passed: 1.2 ACCEPT, 3.1 REJECT (category), 1.1 REJECT (duplicate); randint {1,2,3,4} inclusive OK")


### College Board pseudocode trace

Same rule as https://pages.opencodingsociety.com/python/boolean/2027 Section 3:

```
hasProductName <- (productName != "")
validCategory <- (category = "Auto Racing") OR (category = "Drag Racing")
hasSpecNumber <- (specNumber != "")
hasEffectiveDate <- (effectiveDate != "")
duplicateSpec <- (specNumber = "1.1") OR (specNumber = "2.1")
isValid <- hasProductName AND validCategory AND hasSpecNumber AND hasEffectiveDate AND NOT duplicateSpec
```

Trace: `1.2 / Auto Racing / new` -> T AND T AND T AND T AND NOT F = T -> ACCEPT. `3.1 / Street Car` -> validCategory = F -> F -> REJECT. `1.1 / Drag Racing / duplicate` -> NOT T = F -> REJECT. Python `random.randint(0,2)` indexes `test_records`, `random.choice(test_records)` picks one intact record — same decisions as pseudocode.

---
**Notes for submission:** Lesson: CSP 3.15 Random Values | MCQ 3.15: 4/4 | Popcorn Hacks 1-3 completed and copied to notebook: yes | Homework tested: deterministic 3-record validator (ACCEPT/REJECT) + 5 randomized draws via random.choice + 200-draw randint range check.
